# Blended forecast with weights per forecast hour

Station 06123, group 15. Uses the same input files as `forecast_evaluation.ipynb`:

* `dmi_06123_hourly_qc.csv`: measured hourly GHI (from `data_QC_check.ipynb`)
* `Forecast data/06123_*.csv`: the 8 NWP models for our station
* `DTU forecast/dtu_*.csv`: the DTU forecasts for the competition

**Method**

1. Every forecast value is compared with the measured GHI for the same hour. Only daytime hours are used (sun elevation > 0).
2. For every **forecast hour** (horizon 1, 2, 3, … h after the forecast was issued) and every model, the error is collected over all forecast files. Each model gets a weight from its error at that forecast hour. By default the weight is `1 / MSE`, so a model with half the RMSE gets 4× the weight.
3. Single hours are noisy (few daytime pairs), so the error is smoothed over ±`WINDOW` neighbouring hours.
4. When the weights are used on a new forecast: if a model has **no value** for that hour, its weight is **0** and the other weights are scaled up, so they still add up to 1. If a model has too few pairs at a horizon (e.g. Harmonie after ~54 h), its weight is 0 there.
5. The weights are tested on forecast files they were **not** fitted on, and also "walk-forward": every file is forecast with weights learnt only from what was measured before it was issued. That is how the weights are used for the next forecast.
6. The final weights (fitted on all data) are applied to the DTU file → `Group15.csv`.

**Changing the weights:** see the *Settings* cell: `WEIGHT_METHOD`, `WINDOW`, `EXCLUDE_MODELS`, `MANUAL_FACTORS`, or edit `hourly_weights.csv` by hand and set `WEIGHTS_FILE`.

Just run all cells (Run All). Figures are saved as PNG in the folder `figures_hourly_weights/`.

In [ ]:
import glob
import os
import re
import json
import itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pvlib
from IPython.display import display

plt.rcParams['figure.dpi'] = 100
os.makedirs('figures_hourly_weights', exist_ok=True)

def save(name):
    plt.savefig(os.path.join('figures_hourly_weights', name), dpi=200, bbox_inches='tight')

## Settings
Everything you might want to change is here.

In [ ]:
station_id = '06123'
latitude = 55.2493          # same values as in data_QC_check.ipynb / forecast_evaluation.ipynb
longitude = 9.8899
group_number = 15

models = ['dmi_harmonie_arome_europe', 'metno_nordic', 'ecmwf_ifs025', 'ecmwf_aifs025_single',
          'gfs_global', 'ecmwf_ifs', 'dmi_seamless', 'gfs_seamless']

measurement_file = 'dmi_' + station_id + '_hourly_qc.csv'
DOWNLOAD_NEW_FILES = True    # get new forecast files from the course GitHub (same as forecast_evaluation.ipynb)

# Forecast files that are known to be bad (also see the automatic check in section 3)
BAD_FILES = ['06123_20261103T1100.csv']
AUTO_EXCLUDE_FLAGGED = True  # also leave out files that the automatic check flags

# ---- How the weights are made ------------------------------------------------
MAX_HORIZON = 168            # weights are made for forecast hours 1 ... 168 (7 days)
WEIGHT_METHOD = 'inverse_mse'   # 'inverse_mse'  : w = 1 / RMSE²   (default)
                                # 'inverse_rmse' : w = 1 / RMSE    (softer)
                                # 'best_only'    : weight 1 to the best model at each hour, 0 to the rest
                                # 'equal'        : same weight to every model that has data at that hour
WINDOW = 3                   # smooth the error over ±WINDOW forecast hours (0 = every hour on its own)
MIN_PAIRS = 15               # a model needs at least this many daytime pairs (inside the window) to get a weight
DROP_IDENTICAL = True        # leave out a model that is an exact copy of another one (e.g. a _seamless model)

EXCLUDE_MODELS = []          # e.g. ['gfs_seamless'] -> weight 0 everywhere
MANUAL_FACTORS = {}          # multiply a model's weight, e.g. {'gfs_global': 0.5, 'ecmwf_ifs': 2}
WEIGHTS_FILE = None          # e.g. 'hourly_weights_edited.csv' -> use these weights instead (edit the saved csv)

TEST_FRACTION = 0.3          # newest 30 % of the forecast files are used for testing

# ---- Competition ---------------------------------------------------------------
PERIOD_START = '2026-11-28 00:00'   # first hour of the competition period (UTC, hour ending)
PERIOD_END = '2026-12-01 23:00'     # last hour (28 Nov - 1 Dec = 4 days)
COMPETITION_HORIZON = 96            # used for the TEST file if the competition file is not there yet

location = pvlib.location.Location(latitude, longitude)

## 1. Get the newest forecast files
Same download as in `forecast_evaluation.ipynb`. Files that are already in the folder are skipped. If there is no internet, the files already in the folder are used.

In [ ]:
from urllib.request import urlopen
from urllib.error import HTTPError, URLError

REPO = 'AdamRJensen/solar-resource-course-2026'
RAW_URL = f'https://raw.githubusercontent.com/{REPO}/refs/heads/main/forecast-data/'
API_URL = f'https://api.github.com/repos/{REPO}/git/trees/main?recursive=1'

def download_forecasts(prefix, folder):
    os.makedirs(folder, exist_ok=True)
    with urlopen(API_URL, timeout=30) as response:
        tree = json.load(response)['tree']
    names = sorted(os.path.basename(t['path']) for t in tree
                   if t['path'].startswith('forecast-data/' + prefix) and t['path'].endswith('.csv'))
    new = 0
    for name in names:
        path = os.path.join(folder, name)
        if os.path.exists(path):
            continue
        try:
            with urlopen(RAW_URL + name, timeout=30) as response:
                content = response.read()
        except HTTPError:
            continue
        with open(path + '.part', 'wb') as f:
            f.write(content)
        os.replace(path + '.part', path)
        new += 1
    print(f'{prefix}*: {len(names)} files on GitHub, {new} new files saved in "{folder}/"')

if DOWNLOAD_NEW_FILES:
    try:
        download_forecasts(station_id + '_', 'Forecast data')
        download_forecasts('dtu_', 'DTU forecast')
    except Exception as error:
        print('Download not possible (', error, ') - using the files already in the folder')

forecast_files = sorted(set(glob.glob(station_id + '_*.csv') + glob.glob('Forecast data/' + station_id + '_*.csv')))
dtu_files = sorted(set(glob.glob('dtu_*.csv') + glob.glob('DTU forecast/dtu_*.csv')))
print('Station forecast files:', len(forecast_files), '  DTU files:', len(dtu_files))

## 2. Load the measurements and the forecasts
The forecast horizon = hours between the issue time (from the file name) and the forecast timestamp. Timestamps are hour-ending (right-labeled), like the measurements.

In [ ]:
measured = pd.read_csv(measurement_file, index_col=0)
measured.index = pd.to_datetime(measured.index, utc=True)
measured = measured['ghi'] if 'ghi' in measured.columns else measured.iloc[:, 0]
measured = measured.sort_index()
print('Measurements from', measured.index[0], 'to', measured.dropna().index[-1])

def issue_time_from_name(path):
    match = re.search(r'(\d{8}T\d{4})', os.path.basename(path))
    return pd.to_datetime(match.group(1), format='%Y%m%dT%H%M', utc=True) if match else None

def read_forecast(path):
    df = pd.read_csv(path, index_col=0)
    df.index = pd.to_datetime(df.index, utc=True)
    df = df.apply(pd.to_numeric, errors='coerce')
    return df.reindex(columns=models)

parts = []
for path in forecast_files:
    issue = issue_time_from_name(path)
    if issue is None:
        continue
    df = read_forecast(path)
    df.index.name = 'time'
    long = df.reset_index().melt(id_vars='time', var_name='model', value_name='forecast')
    long['issue_time'] = issue
    long['file'] = os.path.basename(path)
    parts.append(long)

fc = pd.concat(parts, ignore_index=True)
fc['horizon'] = ((fc['time'] - fc['issue_time']) / pd.Timedelta(hours=1)).round().astype(int)
fc = fc[(fc['horizon'] >= 1) & (fc['horizon'] <= MAX_HORIZON)].copy()   # only real forecast hours

# Sun position and clear sky in the middle of each hour (timestamps are hour-ending)
times = pd.DatetimeIndex(sorted(fc['time'].unique()))
mid = times - pd.Timedelta(minutes=30)
sun = location.get_solarposition(mid)
clear = location.get_clearsky(mid)
sun_table = pd.DataFrame({'elevation': sun['elevation'].values, 'ghi_clear': clear['ghi'].values}, index=times)

fc = fc.join(sun_table, on='time')
fc['measured'] = fc['time'].map(measured)
print('Forecast rows:', len(fc), '  files:', fc['file'].nunique(),
      '  issue times from', fc['issue_time'].min(), 'to', fc['issue_time'].max())

## 3. Check the forecast files
* A file is **flagged** if a model forecasts sun at night (sun elevation < −5°, GHI > 20 W/m²) in 3 or more hours. That is typical for a time shift in the file. Hours far above clear sky (GHI > 1.3 · clear sky + 50) are counted and shown too.
* Models that are exact copies of another model are found (they would otherwise be counted twice in the weights).

In [ ]:
night_bad = (fc['elevation'] < -5) & (fc['forecast'] > 20)           # sun at night -> time shift
over_clear = (fc['elevation'] > 0) & (fc['forecast'] > 1.3 * fc['ghi_clear'] + 50)
check = pd.DataFrame({'night_hours': night_bad.groupby(fc['file']).sum(),
                      'above_clear_sky_hours': over_clear.groupby(fc['file']).sum()})
flagged = check.index[check['night_hours'] >= 3].tolist()
print('Files with sun forecast at night (>= 3 hours):', flagged if flagged else 'none')
print('Hours far above clear sky (only shown, not used to remove files):',
      int(check['above_clear_sky_hours'].sum()), 'in', int((check['above_clear_sky_hours'] > 0).sum()), 'files')

if len(flagged) > 0.2 * len(check):
    print(f'WARNING: {len(flagged)} of {len(check)} files flagged - that looks like a problem with the check, '
          'not with the files. They are NOT removed; check latitude/longitude.')
    flagged = []
exclude_files = set(BAD_FILES) | (set(flagged) if AUTO_EXCLUDE_FLAGGED else set())
fc = fc[~fc['file'].isin(exclude_files)].copy()
print('Left out:', sorted(exclude_files))
print('Files used:', fc['file'].nunique())

# Exact copies
wide_all = fc.pivot_table(index=['issue_time', 'time'], columns='model', values='forecast').reindex(columns=models)
rows = []
for a, b in itertools.combinations(models, 2):
    both = wide_all[[a, b]].dropna()
    if len(both) > 0:
        rows.append({'model_a': a, 'model_b': b, 'pairs': len(both),
                     'identical_%': 100 * ((both[a] - both[b]).abs() < 0.5).mean()})
same = pd.DataFrame(rows).sort_values('identical_%', ascending=False)
display(same.head(6).round(1))

copies = []
if DROP_IDENTICAL:
    for _, r in same[same['identical_%'] >= 99.9].iterrows():
        if r['model_a'] not in copies and r['model_b'] not in copies:
            copies.append(r['model_b'])
print('Exact copies left out of the weights:', copies if copies else 'none')

In [ ]:
# Daytime pairs with a measurement: these are used to score and to make the weights
pairs = fc[(fc['elevation'] > 0) & fc['measured'].notna() & fc['forecast'].notna()].copy()
pairs['error'] = pairs['forecast'] - pairs['measured']
print('Daytime forecast/measurement pairs:', len(pairs))
if len(pairs) == 0:
    raise ValueError('No pairs - the measurements do not reach past the forecast issue times. Re-run data_QC_check.ipynb.')

# How many pairs each model has at each forecast hour
count = pairs.groupby(['horizon', 'model']).size().unstack().reindex(index=range(1, MAX_HORIZON + 1), columns=models).fillna(0)

fig, ax = plt.subplots(figsize=(13, 4))
for m in models:
    ax.plot(count.index, count[m], label=m, linewidth=1)
ax.set_xlabel('Forecast horizon [hours]')
ax.set_ylabel('Daytime pairs')
ax.set_title('Number of daytime forecast/measurement pairs per forecast hour')
ax.legend(fontsize=7, ncol=4)
for d in range(24, MAX_HORIZON + 1, 24):
    ax.axvline(d, color='grey', linewidth=0.5, linestyle=':')
plt.tight_layout(); save('01_pairs_per_horizon.png'); plt.show()

## 4. Error of every model at every forecast hour
Thin lines: every single forecast hour. Thick lines: smoothed over ±`WINDOW` hours (this is what the weights are made from).

In [ ]:
def error_tables(p, window=WINDOW):
    """SSE, count and sum of errors per forecast hour (rows) and model (columns), smoothed over ±window hours."""
    idx = range(1, MAX_HORIZON + 1)
    g = p.groupby(['horizon', 'model'])['error']
    sse = g.apply(lambda e: (e ** 2).sum()).unstack().reindex(index=idx, columns=models).fillna(0)
    n = g.size().unstack().reindex(index=idx, columns=models).fillna(0)
    s = g.sum().unstack().reindex(index=idx, columns=models).fillna(0)
    if window > 0:
        roll = lambda t: t.rolling(2 * window + 1, center=True, min_periods=1).sum()
        sse, n, s = roll(sse), roll(n), roll(s)
    return sse, n, s

sse0, n0, s0 = error_tables(pairs, window=0)
sse_w, n_w, s_w = error_tables(pairs, window=WINDOW)
rmse_raw = np.sqrt(sse0 / n0.where(n0 > 0))
rmse_smooth = np.sqrt(sse_w / n_w.where(n_w >= MIN_PAIRS))
bias_smooth = s_w / n_w.where(n_w >= MIN_PAIRS)

colors = dict(zip(models, plt.cm.tab10.colors))
fig, axes = plt.subplots(2, 1, figsize=(13, 8), sharex=True)
for m in models:
    axes[0].plot(rmse_raw.index, rmse_raw[m], color=colors[m], linewidth=0.5, alpha=0.4)
    axes[0].plot(rmse_smooth.index, rmse_smooth[m], color=colors[m], linewidth=2, label=m)
    axes[1].plot(bias_smooth.index, bias_smooth[m], color=colors[m], linewidth=2, label=m)
axes[0].set_ylabel('RMSE [W/m²]')
axes[0].set_title(f'RMSE and bias per forecast hour (thick = smoothed ±{WINDOW} h)')
axes[1].set_ylabel('Bias [W/m²]')
axes[1].axhline(0, color='black', linewidth=0.8)
axes[1].set_xlabel('Forecast horizon [hours]')
axes[0].legend(fontsize=7, ncol=4)
plt.tight_layout(); save('02_rmse_bias_per_horizon.png'); plt.show()

print('RMSE [W/m²] per forecast day (all hours of that day together):')
pairs['day_ahead'] = (pairs['horizon'] - 1) // 24 + 1
day_rmse = pairs.groupby(['day_ahead', 'model'])['error'].apply(lambda e: np.sqrt((e ** 2).mean())).unstack().reindex(columns=models)
display(day_rmse.round(1))

## 5. Making the weights and using them

* `make_weights(p)`: weights per forecast hour (rows) and model (columns) from the pairs `p`. Every row adds up to 1.
* `apply_weights(table, w)`: the blended forecast. For every hour it takes the weights of that forecast hour; a model **without a value gets weight 0** and the others are scaled up. If no model with a value has a weight (e.g. a horizon with no training data), the plain mean of the available models is used.

In [ ]:
def make_weights(p, method=WEIGHT_METHOD, window=WINDOW, min_pairs=MIN_PAIRS):
    sse, n, _ = error_tables(p, window=window)
    mse = (sse / n).where(n >= min_pairs)
    if method == 'inverse_mse':
        w = 1 / mse
    elif method == 'inverse_rmse':
        w = 1 / np.sqrt(mse)
    elif method == 'best_only':
        w = mse.eq(mse.min(axis=1), axis=0).astype(float).where(mse.notna())
    elif method == 'equal':
        w = mse.notna().astype(float).where(mse.notna())
    else:
        raise ValueError('Unknown WEIGHT_METHOD: ' + method)
    w = w.replace([np.inf, -np.inf], np.nan).fillna(0)
    for m in set(EXCLUDE_MODELS) | set(copies):
        if m in w.columns:
            w[m] = 0.0
    for m, factor in MANUAL_FACTORS.items():
        if m in w.columns:
            w[m] = w[m] * factor
    total = w.sum(axis=1)
    return w.div(total.where(total > 0), axis=0).fillna(0)

def apply_weights(table, w):
    """table: one row per hour, columns = models (+ 'horizon'). Returns the blended forecast."""
    F = table[models].to_numpy(dtype=float)
    h = table['horizon'].clip(1, MAX_HORIZON).to_numpy(dtype=int)
    W = w.reindex(index=range(1, MAX_HORIZON + 1), columns=models).fillna(0).to_numpy()[h - 1]
    available = ~np.isnan(F)
    W = W * available                                    # missing forecast -> weight 0
    total = W.sum(axis=1)
    no_weight = (total == 0) & available.any(axis=1)     # fallback: plain mean of what is there
    W[no_weight] = available[no_weight]
    total = W.sum(axis=1)
    blend = np.where(total > 0, np.nansum(np.nan_to_num(F) * W, axis=1) / np.where(total > 0, total, 1), np.nan)
    return pd.Series(np.clip(blend, 0, None), index=table.index)

def rmse(e):
    e = pd.Series(e).dropna()
    return float(np.sqrt((e ** 2).mean())) if len(e) else np.nan

# One row per (issue time, time) with all models side by side - daytime with a measurement
table = fc[(fc['elevation'] > 0) & fc['measured'].notna()].pivot_table(
    index=['issue_time', 'time'], columns='model', values='forecast').reindex(columns=models)
first = fc.groupby(['issue_time', 'time'])[['measured', 'horizon', 'ghi_clear']].first()
table = table.join(first, how='inner')
table = table[table[models].notna().any(axis=1)]
print('Hours to score:', len(table))

## 6. Test on forecast files the weights have not seen
The oldest files are used to make the weights (training), the newest `TEST_FRACTION` are used for testing. The per-hour blend is compared with every single model, the plain mean of the models, and the "mean of all" method from `forecast_evaluation.ipynb`.

In [ ]:
issues = np.sort(table.index.get_level_values('issue_time').unique())
split_time = issues[int(len(issues) * (1 - TEST_FRACTION))] if len(issues) >= 6 else issues[-1]
p_issue = pairs['issue_time']
train_pairs = pairs[p_issue < split_time]
t_issue = table.index.get_level_values('issue_time')
test = table[t_issue >= split_time].copy()
print('Training files:', train_pairs['issue_time'].nunique(), '  Test files:', test.index.get_level_values('issue_time').nunique())
print('Training issued before', split_time)

w_train = make_weights(train_pairs)
equal_models = [m for m in models if m not in set(EXCLUDE_MODELS) | set(copies)]
w_equal = make_weights(train_pairs, method='equal')

test['blend_hourly'] = apply_weights(test, w_train)
test['mean_models'] = apply_weights(test, w_equal)
# 'mean of all' from forecast_evaluation.ipynb: the 5 models that reach 96 h, equal weights
mean5 = [m for m in ['ecmwf_ifs025', 'ecmwf_aifs025_single', 'gfs_global', 'ecmwf_ifs', 'dmi_seamless'] if m in models]
test['mean_of_5'] = test[mean5].mean(axis=1)

candidates = models + ['mean_models', 'mean_of_5', 'blend_hourly']
res = []
for c in candidates:
    e = test[c] - test['measured']
    ok = test[c].notna()
    res.append({'forecast': c, 'hours': int(ok.sum()), 'RMSE': rmse(e), 'bias': e.mean(), 'MAE': e.abs().mean(),
                'RMSE_first_96h': rmse(e[test['horizon'] <= 96])})
results = pd.DataFrame(res).set_index('forecast').sort_values('RMSE_first_96h')
display(results.round(1))
print('Note: models that stop early (Harmonie, MetNo) are only scored on the hours they have, so compare them per horizon below.')

In [ ]:
# RMSE and bias per forecast hour on the TEST files
def per_horizon(df, col, window=WINDOW):
    e = (df[col] - df['measured']).dropna()
    h = df.loc[e.index, 'horizon']
    sse = (e ** 2).groupby(h).sum().reindex(range(1, MAX_HORIZON + 1)).fillna(0)
    n = e.groupby(h).size().reindex(range(1, MAX_HORIZON + 1)).fillna(0)
    s = e.groupby(h).sum().reindex(range(1, MAX_HORIZON + 1)).fillna(0)
    if window > 0:
        sse, n, s = [t.rolling(2 * window + 1, center=True, min_periods=1).sum() for t in (sse, n, s)]
    return np.sqrt(sse / n.where(n > 0)), s / n.where(n > 0)

fig, axes = plt.subplots(2, 1, figsize=(13, 8), sharex=True)
for c in models:
    r, b = per_horizon(test, c)
    axes[0].plot(r.index, r, color=colors[c], linewidth=1, alpha=0.7, label=c)
    axes[1].plot(b.index, b, color=colors[c], linewidth=1, alpha=0.7)
for c, style in [('mean_of_5', dict(color='grey', linewidth=2.5, linestyle='--')),
                 ('blend_hourly', dict(color='black', linewidth=2.5))]:
    r, b = per_horizon(test, c)
    axes[0].plot(r.index, r, label=c, **style)
    axes[1].plot(b.index, b, **style)
axes[0].set_ylabel('RMSE [W/m²]')
axes[0].set_title('Test files: per-hour blend (black) vs. single models and mean of 5 (dashed)')
axes[1].set_ylabel('Bias [W/m²]')
axes[1].axhline(0, color='black', linewidth=0.8)
axes[1].set_xlabel('Forecast horizon [hours]')
axes[0].legend(fontsize=7, ncol=4)
plt.tight_layout(); save('03_test_rmse_bias_blend_vs_models.png'); plt.show()

# Bar chart of the test RMSE for the first 96 h
fig, ax = plt.subplots(figsize=(10, 4))
r96 = results['RMSE_first_96h'].dropna().sort_values()
bar_colors = ['black' if i == 'blend_hourly' else 'grey' if i in ('mean_of_5', 'mean_models') else colors.get(i, 'C0') for i in r96.index]
ax.barh(r96.index, r96.values, color=bar_colors)
ax.invert_yaxis()
ax.set_xlabel('RMSE on test files, forecast hours 1-96 [W/m²]')
ax.set_title('Which forecast is best on the test files?')
for i, v in enumerate(r96.values):
    ax.text(v, i, f' {v:.1f}', va='center', fontsize=8)
plt.tight_layout(); save('04_test_rmse_bar.png'); plt.show()

## 7. Walk-forward test: weights used on the next forecast
This is how the method would be used for real. For every test file, the weights are made **only from hours that were already measured when that file was issued**. Then they are applied to that file. The weights are updated for every new file.

In [ ]:
test_issues = np.sort(test.index.get_level_values('issue_time').unique())
wf_parts = []
for issue in test_issues:
    known = pairs[pairs['time'] <= issue]            # only what was measured at the issue time
    if known['issue_time'].nunique() < 3:
        continue
    w = make_weights(known)
    one = test.xs(issue, level='issue_time', drop_level=False).copy()
    one['blend_walk_forward'] = apply_weights(one, w)
    wf_parts.append(one)
wf = pd.concat(wf_parts)

wf_rows = []
for issue, one in wf.groupby(level='issue_time'):
    one96 = one[one['horizon'] <= 96]
    wf_rows.append({'issue_time': issue,
                    'blend_walk_forward': rmse(one96['blend_walk_forward'] - one96['measured']),
                    'mean_of_5': rmse(one96['mean_of_5'] - one96['measured']),
                    'best_single_(ecmwf_ifs025)': rmse(one96['ecmwf_ifs025'] - one96['measured'])})
wf_scores = pd.DataFrame(wf_rows).set_index('issue_time')

w96 = wf[wf['horizon'] <= 96]
print('Walk-forward, forecast hours 1-96, all test files together:')
summary_wf = pd.Series({
    'blend_walk_forward': rmse(w96['blend_walk_forward'] - w96['measured']),
    'blend_hourly (fixed training weights)': rmse(w96['blend_hourly'] - w96['measured']),
    'mean_of_5': rmse(w96['mean_of_5'] - w96['measured']),
    'ecmwf_ifs025': rmse(w96['ecmwf_ifs025'] - w96['measured'])}, name='RMSE [W/m²]')
display(summary_wf.round(1).to_frame())

fig, ax = plt.subplots(figsize=(13, 4))
for c, style in [('blend_walk_forward', dict(color='black', linewidth=2, marker='o', markersize=3)),
                 ('mean_of_5', dict(color='grey', linestyle='--', marker='s', markersize=3)),
                 ('best_single_(ecmwf_ifs025)', dict(color=colors['ecmwf_ifs025'], marker='^', markersize=3))]:
    ax.plot(wf_scores.index, wf_scores[c], label=c, **style)
ax.set_ylabel('RMSE, hours 1-96 [W/m²]')
ax.set_xlabel('Forecast issue time')
ax.set_title('Walk-forward: RMSE of every test forecast file')
ax.legend(fontsize=8)
plt.tight_layout(); save('05_walk_forward_rmse_per_file.png'); plt.show()

share = (wf_scores['blend_walk_forward'] < wf_scores['mean_of_5']).mean()
print(f'The per-hour blend beats the mean of 5 in {100 * share:.0f} % of the test files.')

## 8. Which settings work best?
Test RMSE (hours 1–96) for different weight methods and smoothing windows, fitted on the training files. Use this to choose `WEIGHT_METHOD` and `WINDOW` in the settings.

In [ ]:
methods_to_try = ['inverse_mse', 'inverse_rmse', 'best_only', 'equal']
windows_to_try = [0, 1, 3, 6, 12]
t96 = test[test['horizon'] <= 96]
grid = pd.DataFrame(index=methods_to_try, columns=windows_to_try, dtype=float)
for meth in methods_to_try:
    for win in windows_to_try:
        w = make_weights(train_pairs, method=meth, window=win)
        grid.loc[meth, win] = rmse(apply_weights(t96, w) - t96['measured'])
grid.columns.name = 'WINDOW (± hours)'
display(grid.round(1))

fig, ax = plt.subplots(figsize=(8, 4))
for meth in methods_to_try:
    ax.plot(windows_to_try, grid.loc[meth], marker='o', label=meth)
ax.axhline(rmse(t96['mean_of_5'] - t96['measured']), color='grey', linestyle='--', label='mean of 5')
ax.set_xlabel('Smoothing window WINDOW [± hours]')
ax.set_ylabel('Test RMSE, hours 1-96 [W/m²]')
ax.set_title('Settings for the weights')
ax.legend(fontsize=8)
plt.tight_layout(); save('06_settings_sensitivity.png'); plt.show()
best = grid.stack().idxmin()
print(f'Best on the test files: WEIGHT_METHOD = {best[0]!r}, WINDOW = {best[1]}  (current: {WEIGHT_METHOD!r}, {WINDOW})')

## 9. Final weights (all data)
The weights are made again from **all** forecast files, saved to `hourly_weights.csv`, and used for the competition.

To change them by hand: open `hourly_weights.csv`, change the numbers (rows = forecast hour, columns = model; the rows do not need to add up to 1, they are scaled), save it as e.g. `hourly_weights_edited.csv` and set `WEIGHTS_FILE = 'hourly_weights_edited.csv'` in the settings.

In [ ]:
if WEIGHTS_FILE:
    final_w = pd.read_csv(WEIGHTS_FILE, index_col=0).reindex(index=range(1, MAX_HORIZON + 1), columns=models).fillna(0)
    final_w = final_w.div(final_w.sum(axis=1).where(final_w.sum(axis=1) > 0), axis=0).fillna(0)
    print('Weights read from', WEIGHTS_FILE)
else:
    final_w = make_weights(pairs)
    final_w.index.name = 'horizon'
    final_w.round(4).to_csv('hourly_weights.csv')
    print('Weights saved to hourly_weights.csv')

empty = final_w.index[final_w.sum(axis=1) == 0]
if len(empty):
    print(f'No weights for forecast hours {empty.min()}-{empty.max()} (not enough data) -> plain mean of the models is used there')

print('Average weight per forecast day:')
by_day = final_w.groupby((final_w.index - 1) // 24 + 1).mean()
by_day.index.name = 'forecast day'
display(by_day.round(2))

In [ ]:
# Stacked weights per forecast hour
fig, ax = plt.subplots(figsize=(13, 4.5))
ax.stackplot(final_w.index, [final_w[m] for m in models], labels=models, colors=[colors[m] for m in models])
for d in range(24, MAX_HORIZON + 1, 24):
    ax.axvline(d, color='white', linewidth=0.8)
ax.set_xlim(1, MAX_HORIZON)
ax.set_ylim(0, 1)
ax.set_xlabel('Forecast horizon [hours]')
ax.set_ylabel('Weight')
ax.set_title(f'Final weights per forecast hour ({WEIGHT_METHOD}, ±{WINDOW} h)')
ax.legend(fontsize=7, ncol=4, loc='upper right')
plt.tight_layout(); save('07_final_weights_stacked.png'); plt.show()

# Heatmap
fig, ax = plt.subplots(figsize=(13, 3.5))
im = ax.pcolormesh(final_w.index, range(len(models)), final_w.T.values, shading='nearest', cmap='viridis')
ax.set_yticks(range(len(models)))
ax.set_yticklabels(models, fontsize=8)
ax.set_xlabel('Forecast horizon [hours]')
ax.set_title('Final weights (heatmap)')
plt.colorbar(im, ax=ax, label='Weight')
plt.tight_layout(); save('08_final_weights_heatmap.png'); plt.show()

# Weight vs RMSE: the link between the two
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
rm_all = np.sqrt(sse_w / n_w.where(n_w >= MIN_PAIRS))
for m in models:
    axes[0].plot(final_w.index, final_w[m], color=colors[m], label=m)
    axes[1].scatter(rm_all[m], final_w[m], s=6, color=colors[m], label=m)
axes[0].set_xlabel('Forecast horizon [hours]'); axes[0].set_ylabel('Weight'); axes[0].set_title('Weights as lines')
axes[1].set_xlabel('Smoothed RMSE at that forecast hour [W/m²]'); axes[1].set_ylabel('Weight'); axes[1].set_title('Lower RMSE -> higher weight')
axes[0].legend(fontsize=7)
plt.tight_layout(); save('09_weights_lines_and_vs_rmse.png'); plt.show()

## 10. Forecast vs. measured for some test files
Every model (thin), the per-hour blend (thick black) and the measurement (thick red) for the first 4 days of a few test files. Change `example_files` to look at other files.

In [ ]:
example_files = [0, len(test_issues) // 2, -1]     # positions in the list of test files
fig, axes = plt.subplots(len(example_files), 1, figsize=(14, 3.6 * len(example_files)))
axes = np.atleast_1d(axes)
for ax, k in zip(axes, example_files):
    issue = test_issues[k]
    one = fc[(fc['issue_time'] == issue) & (fc['horizon'] <= 96)]
    wide = one.pivot_table(index='time', columns='model', values='forecast').reindex(columns=models)
    wide['horizon'] = one.groupby('time')['horizon'].first()
    wide['blend'] = apply_weights(wide, w_train)
    for m in models:
        ax.plot(wide.index, wide[m], color=colors[m], linewidth=0.8, alpha=0.8, label=m)
    ax.plot(wide.index, wide['blend'], color='black', linewidth=2.2, label='per-hour blend')
    meas = one.drop_duplicates('time').set_index('time')
    ax.plot(meas.index, meas['measured'], color='red', linewidth=2, label='measured')
    ax.plot(meas.index, meas['ghi_clear'], color='grey', linestyle=':', label='clear sky')
    ax.set_ylabel('GHI [W/m²]')
    ax.set_title(f'Forecast issued {issue:%d %b %Y %H:%M} UTC')
axes[0].legend(fontsize=7, ncol=6)
plt.tight_layout(); save('10_examples_forecast_vs_measured.png'); plt.show()

In [ ]:
# Scatter: forecast vs. measured on the test files
cols = models + ['blend_hourly']
ncol = 3
nrow = int(np.ceil(len(cols) / ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(12, 3.6 * nrow), sharex=True, sharey=True)
vmax = float(np.nanmax(test[['measured'] + cols].to_numpy())) if len(test) else 500
for ax, c in zip(axes.flatten(), cols):
    ok = test[c].notna()
    ax.scatter(test.loc[ok, 'measured'], test.loc[ok, c], s=3, alpha=0.35,
               color='black' if c == 'blend_hourly' else colors[c])
    ax.plot([0, vmax], [0, vmax], color='red', linewidth=1)
    ax.set_title(f'{c}\nRMSE {rmse(test.loc[ok, c] - test.loc[ok, "measured"]):.1f} W/m²', fontsize=9)
for ax in axes.flatten()[len(cols):]:
    ax.axis('off')
for ax in axes[-1]:
    ax.set_xlabel('Measured [W/m²]')
for ax in axes[:, 0]:
    ax.set_ylabel('Forecast [W/m²]')
plt.tight_layout(); save('11_scatter_test.png'); plt.show()

## 11. Clear, broken-cloud and overcast days
Same day classification as in `forecast_evaluation.ipynb` (daily clear-sky index: > 0.7 clear, 0.3–0.7 broken clouds, < 0.3 overcast). Relative RMSE = RMSE / mean measured GHI, so the day types can be compared.

In [ ]:
meas_df = measured.to_frame('ghi')
meas_df['ghi_clear'] = location.get_clearsky(meas_df.index - pd.Timedelta(minutes=30))['ghi'].values
meas_df['date'] = (meas_df.index - pd.Timedelta(minutes=30)).date
day = meas_df.groupby('date').agg(measured=('ghi', 'sum'), clear=('ghi_clear', 'sum'), n=('ghi', 'count'))
day = day[day['n'] == 24]
day['clearsky_index'] = day['measured'] / day['clear']
day['day_type'] = pd.cut(day['clearsky_index'], bins=[-1, 0.3, 0.7, 10], labels=['overcast', 'broken clouds', 'clear'])

tt = test.copy()
tt['date'] = (tt.index.get_level_values('time') - pd.Timedelta(minutes=30)).date
tt['day_type'] = tt['date'].map(day['day_type'])
tt = tt[tt['day_type'].notna() & (tt['horizon'] <= 96)]
print('Test hours per day type:')
display(tt['day_type'].value_counts())

cols = [m for m in models if m not in copies] + ['mean_of_5', 'blend_hourly']
rel = pd.DataFrame(index=['clear', 'broken clouds', 'overcast'], columns=cols, dtype=float)
for dt in rel.index:
    part = tt[tt['day_type'] == dt]
    if len(part) == 0:
        continue
    for c in cols:
        ok = part[c].notna()
        rel.loc[dt, c] = 100 * rmse(part.loc[ok, c] - part.loc[ok, 'measured']) / part.loc[ok, 'measured'].mean() if ok.any() else np.nan
display(rel.round(1))

fig, ax = plt.subplots(figsize=(13, 4.5))
x = np.arange(len(rel.index))
width = 0.8 / len(cols)
for i, c in enumerate(cols):
    color = 'black' if c == 'blend_hourly' else 'grey' if c == 'mean_of_5' else colors[c]
    ax.bar(x + i * width - 0.4 + width / 2, rel[c], width, label=c, color=color)
ax.set_xticks(x)
ax.set_xticklabels(rel.index)
ax.set_ylabel('Relative RMSE [%]')
ax.set_title('Test files, hours 1-96: error per day type')
ax.legend(fontsize=7, ncol=4)
plt.tight_layout(); save('12_day_types_relative_rmse.png'); plt.show()

## 12. Competition: blended forecast for DTU
The newest DTU file that covers `PERIOD_START` – `PERIOD_END` is used. The final weights are applied by forecast hour (from that file's issue time). If no DTU file covers the period yet, a **TEST** file with the first 4 days after the newest issue time is made instead (`Group15_TEST.csv`).

Output: first column date/time like `2026-11-29 02:00+00:00`, second column the blended GHI.

In [ ]:
period_start = pd.Timestamp(PERIOD_START, tz='UTC')
period_end = pd.Timestamp(PERIOD_END, tz='UTC')

dtu_info = []
for path in dtu_files:
    issue = issue_time_from_name(path)
    if issue is None:
        continue
    df = read_forecast(path)
    covers = df.index.min() <= period_start and df.index.max() >= period_end
    dtu_info.append((issue, path, covers, df))

good = [d for d in dtu_info if d[2] and d[0] < period_start]
if good:
    dtu_issue, dtu_path, _, dtu = max(good, key=lambda d: d[0])
    competition = True
else:
    if not dtu_info:
        raise FileNotFoundError('No DTU forecast files found in "DTU forecast/"')
    dtu_issue, dtu_path, _, dtu = max(dtu_info, key=lambda d: d[0])
    competition = False
print('DTU file:', dtu_path, '  issued', dtu_issue, '  covers', dtu.index.min(), 'to', dtu.index.max())

missing = [m for m in models if dtu[m].isna().all()]
if missing:
    print('Models not in the DTU file (weight 0, others scaled up):', missing)

dtu_table = dtu.copy()
dtu_table['horizon'] = ((dtu_table.index - dtu_issue) / pd.Timedelta(hours=1)).round().astype(int)
dtu_table = dtu_table[dtu_table['horizon'] >= 1]
dtu_table['blend'] = apply_weights(dtu_table, final_w)

if competition:
    out = dtu_table.loc[period_start:period_end]
    out_file = f'Group{group_number}.csv'
else:
    out = dtu_table[dtu_table['horizon'] <= COMPETITION_HORIZON]
    out_file = f'Group{group_number}_TEST.csv'
    print(f'\nNo DTU file covers {PERIOD_START} - {PERIOD_END} yet -> TEST file only. Run again when the competition file is there.')

submission = pd.DataFrame({'time': out.index.strftime('%Y-%m-%d %H:%M+00:00'), 'ghi': out['blend'].round(1).values})
submission.to_csv(out_file, index=False)
print(f'\nSaved {out_file}: {len(submission)} hours from {submission["time"].iloc[0]} to {submission["time"].iloc[-1]}'
      f' - NaN hours: {int(submission["ghi"].isna().sum())}')
display(submission.head(12))

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True, gridspec_kw={'height_ratios': [2, 1]})
for m in models:
    if dtu[m].notna().any():
        axes[0].plot(out.index, out[m], color=colors[m], linewidth=0.9, alpha=0.8, label=m)
axes[0].plot(out.index, out['blend'], color='black', linewidth=2.5, label='per-hour blend (submitted)')
axes[0].set_ylabel('GHI [W/m²]')
axes[0].set_title(f'DTU forecast issued {dtu_issue:%d %b %H:%M} UTC -> {out_file}')
axes[0].legend(fontsize=7, ncol=5)

w_used = final_w.reindex(out['horizon'].clip(1, MAX_HORIZON).values)
w_used = w_used.mul(out[models].notna().values)                  # missing forecast -> 0
w_used = w_used.div(w_used.sum(axis=1).where(w_used.sum(axis=1) > 0), axis=0).fillna(0)
axes[1].stackplot(out.index, [w_used[m].values for m in models], colors=[colors[m] for m in models])
axes[1].set_ylabel('Weight used')
axes[1].set_ylim(0, 1)
axes[1].set_xlabel('Time [UTC]')
plt.tight_layout(); save('13_dtu_competition_forecast.png'); plt.show()
print('Figures saved in figures_hourly_weights/')